In [2]:
import torch
from torch import nn, optim
import torchvision
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader
import numpy as np
from math import isqrt

In [3]:
# IMage
D_image = 96
N_CHANNELS = 3

PATCH_SIZE = 8
D_patch = (PATCH_SIZE**2) * N_CHANNELS # 192
N_PATCHES = (D_image**2) // (PATCH_SIZE**2) # 144
N_ROWS = isqrt(N_PATCHES)

# Encoder
D = 192
N_HEADS = 3
Dk = D // N_HEADS # 64
D_mlp = 4*D

# Decoder
D_decoder = D # normally D//2 but ViT-Tiny with 1 Transformer block is already so shallow
# D == D_decoder means I can use the same positional embeddings
D_decoder_mlp = 4*D_decoder

BATCH_SIZE = 256

In [4]:
# Transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4467, 0.4398, 0.4066), (0.2603, 0.2566, 0.2713))
])
# Download and load datasets
supervised_trainset = torchvision.datasets.STL10(
    root='./data', split='train', download=True, transform=transform
)
testset = torchvision.datasets.STL10(
    root='./data', split='test', download=True, transform=transform
)
unlabeled_set = torchvision.datasets.STL10(
    root='./data', split='unlabeled', download=True, transform=transform
)
# DataLoaders
# supervised_trainloader = DataLoader(
#     supervised_trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
# )
testloader = DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)
ssl_trainloader = DataLoader(
    unlabeled_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)

val_size = 500 # 10%
train_size = len(supervised_trainset) - val_size
generator = torch.Generator().manual_seed(12) # reproducible split
train_subset, val_subset = random_split(
    supervised_trainset, [train_size, val_size], generator=generator
)
supervised_trainloader = DataLoader(
    train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)
supervised_valloader = DataLoader(
    val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

In [5]:
device = torch.device('cuda:0')

In [6]:
# My long-implementation
# Calculate our 2D sine-cosine position embeddings
D_pos = D // 2 # 96
# I need to make sure the way I view and unview the patches is the same
N_ROWS = isqrt(N_PATCHES)
row_pos = np.zeros((N_ROWS, D_pos))
col_pos = np.zeros((N_ROWS, D_pos))

i = np.arange(D_pos // 2) # half sine, half cosine
denominators = 10000 ** (2 * i / D_pos)

for row in range(0, N_ROWS):
    row_pos[row, 0::2] = np.sin(row / denominators) # even inddicies
    row_pos[row, 1::2] = np.cos(row / denominators) # odd indicies

for col in range(0, N_ROWS):
    col_pos[col, 0::2] = np.sin(col / denominators) # even inddicies
    col_pos[col, 1::2] = np.cos(col / denominators) # odd indicies

pos_embeddings = np.zeros((N_PATCHES, D))
for row in range(0, N_ROWS):
    for col in range(0, N_ROWS):
        pos_embeddings[row*N_ROWS + col, :] = np.concatenate([row_pos[row, :], col_pos[col, :]])
pos_embeddings = torch.tensor(pos_embeddings, dtype=torch.float32)
pos_embeddings = pos_embeddings.to(device)
pos_embeddings
assert pos_embeddings.device == torch.device('cuda:0'), f"pos_embeddings is on {pos_embeddings.device}"

In [7]:
class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        ### ENCODER

        self.img2enc_projection = nn.Linear(D_patch, D)

        # Block 1
        self.norm1a = nn.LayerNorm(D)
        self.msa1 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm1b = nn.LayerNorm(D)
        self.mlp1a = nn.Linear(D, D_mlp)
        self.mlp1b = nn.Linear(D_mlp, D)

        # Block 2
        self.norm2a = nn.LayerNorm(D)
        self.msa2 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm2b = nn.LayerNorm(D)
        self.mlp2a = nn.Linear(D, D_mlp)
        self.mlp2b = nn.Linear(D_mlp, D)

        # Block 3
        self.norm3a = nn.LayerNorm(D)
        self.msa3 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm3b = nn.LayerNorm(D)
        self.mlp3a = nn.Linear(D, D_mlp)
        self.mlp3b = nn.Linear(D_mlp, D)

        ### DECODER (just single transformer block)
        self.masked_embedding = nn.Parameter(torch.randn(1, 1, D_decoder))
        self.enc2dec_projection = nn.Linear(D, D_decoder)
        self.decoder_norm1 = nn.LayerNorm(D_decoder)
        self.decoder_msa = nn.MultiheadAttention(D_decoder, N_HEADS, batch_first=True)
        self.decoder_norm2 = nn.LayerNorm(D_decoder)
        self.decoder_mlp1 = nn.Linear(D_decoder, D_decoder_mlp)
        self.decoder_mlp2 = nn.Linear(D_decoder_mlp, D_decoder)
        self.dec2img_projection = nn.Linear(D_decoder, D_patch)

    def forward(self, x):
        # x is (B, C, H, W) = (B, 3, 96, 96)
        B, C, H, W = x.shape

        # Make (B, 3, 12, 12, 8, 8)
        patches = x.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        assert patches.shape == (B, N_CHANNELS, N_ROWS, N_ROWS, PATCH_SIZE, PATCH_SIZE), patches.shape 
        
        # Make (B, 12, 12, 3, 8, 8)
        x = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        assert x.shape == (B, N_ROWS, N_ROWS, N_CHANNELS, PATCH_SIZE, PATCH_SIZE), x.shape

        x = x.view(B, N_ROWS, N_ROWS, -1)
        assert x.shape == (B, N_ROWS, N_ROWS, D_patch)

        SEQ = N_PATCHES # N_ROWS ** 2

        x = x.view(B, SEQ, D_patch)
        assert x.shape == (B, 144, 192), x.shape
        truth_patches = x

        # Ok we have our embeddigs of the image
        # now we need to add our constant sine-cosine position embeddings
        x = x + pos_embeddings

        # Create a mask and apply it
        noise = torch.rand(B, SEQ, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1) # Get a list of sorted ids for each B
        n_keep = int(SEQ * 0.25)
        n_not_keep = SEQ - n_keep
        ids_unmasked = ids_shuffle[:, :n_keep] # Store the ids of first 25% -> (B, n_keep)
        ids_masked = ids_shuffle[:, n_keep:] # Store the ids of last 75% -> (B, n_masked)

        # x_unmasked we want (B, n_keep, 192)
        ind_unmasked_enc = ids_unmasked.unsqueeze(-1).expand(-1, -1, D_patch) # (B, n_keep, D)
        ind_unmasked_dec = ids_unmasked.unsqueeze(-1).expand(-1, -1, D_decoder) # (B, n_keep, D)

        # For dim=1, keep all dimensions the same but replaces with the column index
        x_unmasked = torch.gather(x, 1, ind_unmasked_enc)
        assert x_unmasked.shape == (B, n_keep, 192), x_unmasked.shape

        # Pass the 25% through the encoder
        # define the encoder
        embeddings = self.img2enc_projection(x_unmasked)
        # Block 1
        x = self.norm1a(embeddings)
        x, _ = self.msa1(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm1b(embeddings)
        x = F.gelu( self.mlp1a(x) )
        x = self.mlp1b(x)
        embeddings = embeddings + x # skip connection
        # Block 2
        x = self.norm2a(embeddings)
        x, _ = self.msa2(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm2b(embeddings)
        x = F.gelu( self.mlp2a(x) )
        x = self.mlp2b(x)
        embeddings = embeddings + x # skip connection
        # Block 3
        x = self.norm3a(embeddings)
        x, _ = self.msa3(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm3b(embeddings)
        x = F.gelu( self.mlp3a(x) )
        x = self.mlp3b(x)
        embeddings = embeddings + x # skip connection

        ## DECODER
        x = self.masked_embedding.repeat(B, N_PATCHES, 1)
        assert x.shape == (B, N_PATCHES, D_decoder), x.shape
        unmasked_embeddings_dec = self.enc2dec_projection(embeddings)
        embeddings_dec = x.clone().scatter(1, ind_unmasked_dec, unmasked_embeddings_dec)
        # embeddings_dec = torch.scatter(x, 1, ind_unmasked_dec, unmasked_embeddings_dec)
        embeddings_dec = embeddings_dec + pos_embeddings
        x = self.decoder_norm1(embeddings_dec)
        x, _ = self.decoder_msa(x, x, x)
        embeddings_dec = embeddings_dec + x
        x = self.decoder_norm2(embeddings_dec)
        x = F.gelu( self.decoder_mlp1(x) )
        x = self.decoder_mlp2(x)
        embeddings_dec = embeddings_dec + x
        y_patches = self.dec2img_projection(embeddings_dec)

        return y_patches, truth_patches, ids_masked
    
    def encode(self, x):
        # x is (B, C, H, W) = (B, 3, 96, 96)
        B, C, H, W = x.shape

        # Make (B, 3, 12, 12, 8, 8)
        patches = x.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        # Make (B, 12, 12, 3, 8, 8)
        x = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        x = x.view(B, N_PATCHES, D_patch)
        x = x + pos_embeddings # Adding pos_embeddings in pixel space for first two models
        embeddings = self.img2enc_projection(x)
        # Block 1
        x = self.norm1a(embeddings)
        x, _ = self.msa1(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm1b(embeddings)
        x = F.gelu( self.mlp1a(x) )
        x = self.mlp1b(x)
        embeddings = embeddings + x # skip connection
        # Block 2
        x = self.norm2a(embeddings)
        x, _ = self.msa2(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm2b(embeddings)
        x = F.gelu( self.mlp2a(x) )
        x = self.mlp2b(x)
        embeddings = embeddings + x # skip connection
        # Block 3
        x = self.norm3a(embeddings)
        x, _ = self.msa3(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm3b(embeddings)
        x = F.gelu( self.mlp3a(x) )
        x = self.mlp3b(x)
        embeddings = embeddings + x # skip connection

        # Average pool across patches to get a single vector per image
        return embeddings.mean(dim=1)  # (B, D)


# D=384 and 6-layer ViT Encoder

In [17]:

# IMage
D_image = 96
N_CHANNELS = 3

PATCH_SIZE = 8
D_patch = (PATCH_SIZE**2) * N_CHANNELS # 192
N_PATCHES = (D_image**2) // (PATCH_SIZE**2) # 144
N_ROWS = isqrt(N_PATCHES)

# Encoder
D = 384
N_HEADS = 3
Dk = D // N_HEADS # 64
D_mlp = 4*D

# Decoder
D_decoder = D # normally D//2 but ViT-Tiny with 1 Transformer block is already so shallow
# D == D_decoder means I can use the same positional embeddings
D_decoder_mlp = 4*D_decoder


### CALCULATE NEW POSITION EMBEDDINGS ###

D_pos = D // 2 # 96
# I need to make sure the way I view and unview the patches is the same
N_ROWS = isqrt(N_PATCHES)
row_pos = np.zeros((N_ROWS, D_pos))
col_pos = np.zeros((N_ROWS, D_pos))

i = np.arange(D_pos // 2) # half sine, half cosine
denominators = 10000 ** (2 * i / D_pos)

for row in range(0, N_ROWS):
    row_pos[row, 0::2] = np.sin(row / denominators) # even inddicies
    row_pos[row, 1::2] = np.cos(row / denominators) # odd indicies

for col in range(0, N_ROWS):
    col_pos[col, 0::2] = np.sin(col / denominators) # even inddicies
    col_pos[col, 1::2] = np.cos(col / denominators) # odd indicies

pos_embeddings = np.zeros((N_PATCHES, D))
for row in range(0, N_ROWS):
    for col in range(0, N_ROWS):
        pos_embeddings[row*N_ROWS + col, :] = np.concatenate([row_pos[row, :], col_pos[col, :]])
pos_embeddings = torch.tensor(pos_embeddings, dtype=torch.float32)
pos_embeddings = pos_embeddings.to(device)
pos_embeddings
assert pos_embeddings.device == torch.device('cuda:0'), f"pos_embeddings is on {pos_embeddings.device}"


### DEFINE THE MODEL ###

class Net(nn.Module):

    def __init__(self):
        super(Net, self).__init__()

        ### ENCODER

        self.img2enc_projection = nn.Linear(D_patch, D)

        # Block 1
        self.norm1a = nn.LayerNorm(D)
        self.msa1 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm1b = nn.LayerNorm(D)
        self.mlp1a = nn.Linear(D, D_mlp)
        self.mlp1b = nn.Linear(D_mlp, D)

        # Block 2
        self.norm2a = nn.LayerNorm(D)
        self.msa2 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm2b = nn.LayerNorm(D)
        self.mlp2a = nn.Linear(D, D_mlp)
        self.mlp2b = nn.Linear(D_mlp, D)

        # Block 3
        self.norm3a = nn.LayerNorm(D)
        self.msa3 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm3b = nn.LayerNorm(D)
        self.mlp3a = nn.Linear(D, D_mlp)
        self.mlp3b = nn.Linear(D_mlp, D)

        # Block 4
        self.norm4a = nn.LayerNorm(D)
        self.msa4 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm4b = nn.LayerNorm(D)
        self.mlp4a = nn.Linear(D, D_mlp)
        self.mlp4b = nn.Linear(D_mlp, D)

        # Block 5
        self.norm5a = nn.LayerNorm(D)
        self.msa5 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm5b = nn.LayerNorm(D)
        self.mlp5a = nn.Linear(D, D_mlp)
        self.mlp5b = nn.Linear(D_mlp, D)

        # Block 6
        self.norm6a = nn.LayerNorm(D)
        self.msa6 = nn.MultiheadAttention(D, N_HEADS, batch_first=True)
        self.norm6b = nn.LayerNorm(D)
        self.mlp6a = nn.Linear(D, D_mlp)
        self.mlp6b = nn.Linear(D_mlp, D)

        ### DECODER (just single transformer block)
        self.masked_embedding = nn.Parameter(torch.randn(1, 1, D_decoder))
        self.enc2dec_projection = nn.Linear(D, D_decoder)
        self.decoder_norm1 = nn.LayerNorm(D_decoder)
        self.decoder_msa = nn.MultiheadAttention(D_decoder, N_HEADS, batch_first=True)
        self.decoder_norm2 = nn.LayerNorm(D_decoder)
        self.decoder_mlp1 = nn.Linear(D_decoder, D_decoder_mlp)
        self.decoder_mlp2 = nn.Linear(D_decoder_mlp, D_decoder)
        self.dec2img_projection = nn.Linear(D_decoder, D_patch)

    def encode(self, x):
        # x is (B, C, H, W) = (B, 3, 96, 96)
        B, C, H, W = x.shape

        # Make (B, 3, 12, 12, 8, 8)
        patches = x.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        assert patches.shape == (B, N_CHANNELS, N_ROWS, N_ROWS, PATCH_SIZE, PATCH_SIZE), patches.shape 
        
        # Make (B, 12, 12, 3, 8, 8)
        x = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        assert x.shape == (B, N_ROWS, N_ROWS, N_CHANNELS, PATCH_SIZE, PATCH_SIZE), x.shape

        SEQ = N_PATCHES # N_ROWS ** 2

        x = x.view(B, SEQ, D_patch)
        assert x.shape == (B, 144, 192), x.shape
        truth_patches = x

        # Create a mask and apply it
        noise = torch.rand(B, SEQ, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1) # Get a list of sorted ids for each B
        n_keep = int(SEQ * 0.25)
        n_not_keep = SEQ - n_keep
        ids_unmasked = ids_shuffle[:, :n_keep] # Store the ids of first 25% -> (B, n_keep)
        ids_masked = ids_shuffle[:, n_keep:] # Store the ids of last 75% -> (B, n_masked)

        # x_unmasked we want (B, n_keep, 192)
        ind_unmasked_enc = ids_unmasked.unsqueeze(-1).expand(-1, -1, D_patch) # (B, n_keep, D_patch)
        ind_unmasked_dec = ids_unmasked.unsqueeze(-1).expand(-1, -1, D_decoder) # (B, n_keep, D)

        # For dim=1, keep all dimensions the same but replaces with the column index
        x_unmasked = torch.gather(x, 1, ind_unmasked_enc)
        assert x_unmasked.shape == (B, n_keep, 192), x_unmasked.shape

        # Pass the 25% through the encoder
        # define the encoder
        embeddings = self.img2enc_projection(x_unmasked)

        # Add our positional embeddings
        # (N_PATCHES, D)
        # embeddings is (B, n_keep, D)
        ind_unmasked_pos = ids_unmasked.unsqueeze(-1).expand(-1, -1, D)
        local_pos_embeddings = torch.gather(pos_embeddings.unsqueeze(0).expand(B, -1, -1), 1, ind_unmasked_pos)
        embeddings = embeddings + local_pos_embeddings

        # Block 1
        x = self.norm1a(embeddings)
        x, _ = self.msa1(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm1b(embeddings)
        x = F.gelu( self.mlp1a(x) )
        x = self.mlp1b(x)
        embeddings = embeddings + x # skip connection
        # Block 2
        x = self.norm2a(embeddings)
        x, _ = self.msa2(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm2b(embeddings)
        x = F.gelu( self.mlp2a(x) )
        x = self.mlp2b(x)
        embeddings = embeddings + x # skip connection
        # Block 3
        x = self.norm3a(embeddings)
        x, _ = self.msa3(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm3b(embeddings)
        x = F.gelu( self.mlp3a(x) )
        x = self.mlp3b(x)
        embeddings = embeddings + x # skip connection
        # Block 4
        x = self.norm4a(embeddings)
        x, _ = self.msa4(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm4b(embeddings)
        x = F.gelu( self.mlp4a(x) )
        x = self.mlp4b(x)
        embeddings = embeddings + x # skip connection
        # Block 5
        x = self.norm5a(embeddings)
        x, _ = self.msa5(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm5b(embeddings)
        x = F.gelu( self.mlp5a(x) )
        x = self.mlp5b(x)
        embeddings = embeddings + x # skip connection
        # Block 6
        x = self.norm6a(embeddings)
        x, _ = self.msa6(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm6b(embeddings)
        x = F.gelu( self.mlp6a(x) )
        x = self.mlp6b(x)
        embeddings = embeddings + x # skip connection

        # Ok now we need to return the meal across the embeddings
        return embeddings.mean(dim=1) # (B, D)

    def forward(self, x):
        # x is (B, C, H, W) = (B, 3, 96, 96)
        B, C, H, W = x.shape

        # Make (B, 3, 12, 12, 8, 8)
        patches = x.unfold(2, PATCH_SIZE, PATCH_SIZE).unfold(3, PATCH_SIZE, PATCH_SIZE)
        assert patches.shape == (B, N_CHANNELS, N_ROWS, N_ROWS, PATCH_SIZE, PATCH_SIZE), patches.shape 
        
        # Make (B, 12, 12, 3, 8, 8)
        x = patches.permute(0, 2, 3, 1, 4, 5).contiguous()
        assert x.shape == (B, N_ROWS, N_ROWS, N_CHANNELS, PATCH_SIZE, PATCH_SIZE), x.shape

        SEQ = N_PATCHES # N_ROWS ** 2

        x = x.view(B, SEQ, D_patch)
        assert x.shape == (B, 144, 192), x.shape
        truth_patches = x

        # Create a mask and apply it
        noise = torch.rand(B, SEQ, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1) # Get a list of sorted ids for each B
        n_keep = int(SEQ * 0.25)
        n_not_keep = SEQ - n_keep
        ids_unmasked = ids_shuffle[:, :n_keep] # Store the ids of first 25% -> (B, n_keep)
        ids_masked = ids_shuffle[:, n_keep:] # Store the ids of last 75% -> (B, n_masked)

        # x_unmasked we want (B, n_keep, 192)
        ind_unmasked_enc = ids_unmasked.unsqueeze(-1).expand(-1, -1, D_patch) # (B, n_keep, D_patch)
        ind_unmasked_dec = ids_unmasked.unsqueeze(-1).expand(-1, -1, D_decoder) # (B, n_keep, D)

        # For dim=1, keep all dimensions the same but replaces with the column index
        x_unmasked = torch.gather(x, 1, ind_unmasked_enc)
        assert x_unmasked.shape == (B, n_keep, 192), x_unmasked.shape

        # Pass the 25% through the encoder
        # define the encoder
        embeddings = self.img2enc_projection(x_unmasked)

        # Add our positional embeddings
        # (N_PATCHES, D)
        # embeddings is (B, n_keep, D)
        ind_unmasked_pos = ids_unmasked.unsqueeze(-1).expand(-1, -1, D)
        local_pos_embeddings = torch.gather(pos_embeddings.unsqueeze(0).expand(B, -1, -1), 1, ind_unmasked_pos)
        embeddings = embeddings + local_pos_embeddings

        # Block 1
        x = self.norm1a(embeddings)
        x, _ = self.msa1(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm1b(embeddings)
        x = F.gelu( self.mlp1a(x) )
        x = self.mlp1b(x)
        embeddings = embeddings + x # skip connection
        # Block 2
        x = self.norm2a(embeddings)
        x, _ = self.msa2(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm2b(embeddings)
        x = F.gelu( self.mlp2a(x) )
        x = self.mlp2b(x)
        embeddings = embeddings + x # skip connection
        # Block 3
        x = self.norm3a(embeddings)
        x, _ = self.msa3(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm3b(embeddings)
        x = F.gelu( self.mlp3a(x) )
        x = self.mlp3b(x)
        embeddings = embeddings + x # skip connection
        # Block 4
        x = self.norm4a(embeddings)
        x, _ = self.msa4(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm4b(embeddings)
        x = F.gelu( self.mlp4a(x) )
        x = self.mlp4b(x)
        embeddings = embeddings + x # skip connection
        # Block 5
        x = self.norm5a(embeddings)
        x, _ = self.msa5(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm5b(embeddings)
        x = F.gelu( self.mlp5a(x) )
        x = self.mlp5b(x)
        embeddings = embeddings + x # skip connection
        # Block 6
        x = self.norm6a(embeddings)
        x, _ = self.msa6(x, x, x)
        embeddings = embeddings + x # skip connection
        x = self.norm6b(embeddings)
        x = F.gelu( self.mlp6a(x) )
        x = self.mlp6b(x)
        embeddings = embeddings + x # skip connection

        ## DECODER
        x = self.masked_embedding.repeat(B, N_PATCHES, 1)
        assert x.shape == (B, N_PATCHES, D_decoder), x.shape
        unmasked_embeddings_dec = self.enc2dec_projection(embeddings)
        embeddings_dec = x.clone().scatter(1, ind_unmasked_dec, unmasked_embeddings_dec)
        # embeddings_dec = torch.scatter(x, 1, ind_unmasked_dec, unmasked_embeddings_dec)
        embeddings_dec = embeddings_dec + pos_embeddings # broadcast along B in pos_embeddings
        x = self.decoder_norm1(embeddings_dec)
        x, _ = self.decoder_msa(x, x, x)
        embeddings_dec = embeddings_dec + x
        x = self.decoder_norm2(embeddings_dec)
        x = F.gelu( self.decoder_mlp1(x) )
        x = self.decoder_mlp2(x)
        embeddings_dec = embeddings_dec + x
        y_patches = self.dec2img_projection(embeddings_dec)

        return y_patches, truth_patches, ids_masked

# net = Net()
# net.to(device)

In [18]:
net = Net()
# net.load_state_dict(torch.load('mae_pretrain_no_aug_0.259_loss.pth'))
# net.load_state_dict(torch.load('mae_pretrain_aug_0.208_192.pth'))
net.load_state_dict(torch.load('mae_pretrain_aug.pth'))
net.to(device)

Net(
  (img2enc_projection): Linear(in_features=192, out_features=384, bias=True)
  (norm1a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (msa1): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
  )
  (norm1b): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (mlp1a): Linear(in_features=384, out_features=1536, bias=True)
  (mlp1b): Linear(in_features=1536, out_features=384, bias=True)
  (norm2a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (msa2): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=384, out_features=384, bias=True)
  )
  (norm2b): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (mlp2a): Linear(in_features=384, out_features=1536, bias=True)
  (mlp2b): Linear(in_features=1536, out_features=384, bias=True)
  (norm3a): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
  (msa3): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableL

In [9]:
# net.eval()
# N_CLASSES = 10
# linear_probe = nn.Linear(D_decoder, N_CLASSES).to(device)
# optimizer_probe = optim.AdamW(linear_probe.parameters(), 1e-3)
# criterion = nn.CrossEntropyLoss()

# for epoch in range(50):
#     running_loss = 0.0
#     for i, data in enumerate(supervised_trainloader):
#         inputs, labels = data[0].to(device), data[1].to(device)
#         with torch.no_grad():
#             reps = net.encode(inputs)
#         logits = linear_probe(reps)
#         loss = criterion(logits, labels)
#         optimizer_probe.zero_grad()
#         loss.backward()
#         optimizer_probe.step()
#         running_loss += loss.item()
#         if i % 10 == 9:
#             print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.6f}')
#             running_loss = 0.0
# print('Finished training')

[1,    10] loss: 0.225999
[2,    10] loss: 0.211256
[3,    10] loss: 0.199064
[4,    10] loss: 0.190874
[5,    10] loss: 0.183341
[6,    10] loss: 0.178410
[7,    10] loss: 0.172716
[8,    10] loss: 0.168239
[9,    10] loss: 0.164391
[10,    10] loss: 0.162621
[11,    10] loss: 0.157387
[12,    10] loss: 0.154979
[13,    10] loss: 0.152554
[14,    10] loss: 0.151631
[15,    10] loss: 0.150640
[16,    10] loss: 0.147179
[17,    10] loss: 0.145315


KeyboardInterrupt: 

In [19]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR, ConstantLR

net.eval()
N_CLASSES = 10
N_EPOCHS = 50

WARMUP_EPOCHS = 5
HOLD_EPOCHS = 20
COSINE_EPOCHS = N_EPOCHS - WARMUP_EPOCHS - HOLD_EPOCHS # 25

linear_probe = nn.Linear(D_decoder, N_CLASSES).to(device)
optimizer_probe = optim.AdamW(linear_probe.parameters(), lr=1e-3, weight_decay=0.0)

"""
The point of warmup is to avoid blowing up early in training, 
when the probe's weights are random and gradients can be large or unstable. 
AdamW especially benefits because its adaptive moments (m, v) need a few steps to stabilize
 — starting at full LR before they've stabilized can push weights in a bad direction.
"""
warmup = LinearLR(optimizer_probe, start_factor=0.01, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer_probe, T_max=N_EPOCHS - WARMUP_EPOCHS)
hold = ConstantLR(optimizer_probe, factor=1.0, total_iters=HOLD_EPOCHS)
scheduler = SequentialLR(optimizer_probe, [warmup, hold, cosine], milestones=[WARMUP_EPOCHS, WARMUP_EPOCHS + HOLD_EPOCHS])

criterion = nn.CrossEntropyLoss()

best_val_acc = 0.0
best_state = None
best_epoch = -1

for epoch in range(N_EPOCHS):
    # --- train ---
    linear_probe.train()
    running_loss = 0.0
    n_batches = 0
    for i, data in enumerate(supervised_trainloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        with torch.no_grad():
            reps = net.encode(inputs)
        logits = linear_probe(reps)
        loss = criterion(logits, labels)

        optimizer_probe.zero_grad()
        loss.backward()
        optimizer_probe.step()

        running_loss += loss.item()
        n_batches += 1

        if i % 10 == 9:
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / (i + 1):.6f}')

    scheduler.step()
    train_loss = running_loss / n_batches

    # --- validate ---
    linear_probe.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in supervised_valloader:
            inputs, labels = data[0].to(device), data[1].to(device)
            reps = net.encode(inputs)
            logits = linear_probe(reps)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total

    current_lr = optimizer_probe.param_groups[0]['lr']
    print(f'epoch {epoch + 1:3d} | train_loss={train_loss:.4f} | '
          f'val_acc={val_acc:.4f} | lr={current_lr:.2e}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        best_state = {k: v.detach().clone() for k, v in linear_probe.state_dict().items()}

# load best-validation weights back into the probe
linear_probe.load_state_dict(best_state)
linear_probe.eval()

print(f'Finished training. Best val_acc={best_val_acc:.4f} at epoch {best_epoch}')

[1,    10] loss: 2.412417
epoch   1 | train_loss=2.4013 | val_acc=0.0720 | lr=2.08e-04
[2,    10] loss: 2.361618
epoch   2 | train_loss=2.3380 | val_acc=0.1420 | lr=4.06e-04
[3,    10] loss: 2.233939
epoch   3 | train_loss=2.1943 | val_acc=0.3040 | lr=6.04e-04
[4,    10] loss: 2.048646
epoch   4 | train_loss=2.0028 | val_acc=0.4340 | lr=8.02e-04
[5,    10] loss: 1.842858
epoch   5 | train_loss=1.8075 | val_acc=0.4800 | lr=1.00e-03
[6,    10] loss: 1.679996
epoch   6 | train_loss=1.6414 | val_acc=0.5240 | lr=1.00e-03
[7,    10] loss: 1.539497
epoch   7 | train_loss=1.5219 | val_acc=0.5480 | lr=1.00e-03
[8,    10] loss: 1.462128
epoch   8 | train_loss=1.4410 | val_acc=0.5720 | lr=1.00e-03
[9,    10] loss: 1.397244
epoch   9 | train_loss=1.3817 | val_acc=0.5780 | lr=1.00e-03
[10,    10] loss: 1.346112
epoch  10 | train_loss=1.3305 | val_acc=0.5860 | lr=1.00e-03
[11,    10] loss: 1.310817
epoch  11 | train_loss=1.2988 | val_acc=0.5740 | lr=1.00e-03
[12,    10] loss: 1.248209
epoch  12 | tr

In [20]:
# Now how do I test it
linear_probe.eval()
correct = 0
total = 0

with torch.no_grad():
    for i, data in enumerate(testloader):
        inputs, labels = data[0].to(device), data[1].to(device)
        reps = net.encode(inputs)
        logits = linear_probe(reps)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f"Test accuracy: {accuracy:.4f}")

Test accuracy: 0.6326
